# D03 — OpenDART 원문 다운로드·추출 (최신 20건 검증)

D02 JSON의 최신 20개 접수번호에 대해 `document.xml`을 호출합니다. 응답 ZIP/XML 저장, 문단·표 추출을 건별로 기록하고 배치 결과 JSON을 생성합니다.

공시 원문은 XML형식 외에 EUC-KR 인코딩의 HTML 형태로도 반환될 수 있어, 태그 대소문자와 문서 형식을 함께 처리합니다. 인증키는 환경변수 또는 `.env`에서만 읽습니다.


In [1]:
from pathlib import Path
import io
import json
import os
import time
import urllib.parse
import urllib.request
import zipfile
from lxml import etree

ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / 'dart_results' / 'disclosures_list_stock_005930.json').is_file()
)
D02_FILE = ROOT / 'dart_results' / 'disclosures_list_stock_005930.json'
OUTPUT_DIR = ROOT / 'dart_results'
API_URL = 'https://opendart.fss.or.kr/api/document.xml'
USER_AGENT = 'D03-document-download/1.0'


def load_api_key() -> str:
    key = os.environ.get('DART_API_KEY')
    if key:
        return key
    for folder in (ROOT, *ROOT.parents):
        env_file = folder / '.env'
        if not env_file.is_file():
            continue
        for line in env_file.read_text(encoding='utf-8-sig').splitlines():
            if line.strip().startswith('DART_API_KEY='):
                return line.split('=', 1)[1].strip().strip('\"\'')
    raise RuntimeError('DART_API_KEY를 환경변수 또는 프로젝트 .env에 설정하세요.')


def clean_text(element) -> str:
    return ' '.join(''.join(element.itertext()).split())


def tag_xpath(tag: str) -> str:
    # XML 대문자 태그와 HTML 소문자 태그를 같은 방식으로 선택합니다.
    return f'.//*[translate(local-name(), "ABCDEFGHIJKLMNOPQRSTUVWXYZ", "abcdefghijklmnopqrstuvwxyz")="{tag.lower()}"]'


d02_data = json.loads(D02_FILE.read_text(encoding='utf-8'))
filings = d02_data['disclosures'][:20]
if not filings:
    raise RuntimeError('D02 JSON에 조회할 공시가 없습니다.')
print('검증 대상:', len(filings), '건')
print('D02 파일:', D02_FILE.relative_to(ROOT))


검증 대상: 20 건
D02 파일: dart_results\disclosures_list_stock_005930.json


In [2]:
# 최신 공시 20건의 원문 ZIP을 접수번호별로 내려받습니다.
download_results = []
api_key = load_api_key()

for index, filing in enumerate(filings, start=1):
    receipt_no = filing['rcept_no']
    params = urllib.parse.urlencode({
        'crtfc_key': api_key,
        'rcept_no': receipt_no,
    })
    request = urllib.request.Request(
        f'{API_URL}?{params}',
        headers={'User-Agent': USER_AGENT},
        method='GET',
    )
    result = {
        'rcept_no': receipt_no,
        'rcept_dt': filing.get('rcept_dt'),
        'report_nm': filing.get('report_nm', '').strip(),
        'http_status': None,
        'zip_file': None,
        'download_error': None,
    }

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            result['http_status'] = response.status
            payload = response.read()
        if not zipfile.is_zipfile(io.BytesIO(payload)):
            result['download_error'] = '응답이 ZIP 파일이 아닙니다.'
        else:
            zip_path = OUTPUT_DIR / f'D03_document_{receipt_no}.zip'
            zip_path.write_bytes(payload)
            result['zip_file'] = str(zip_path.relative_to(ROOT))
    except Exception as exc:
        # 예외 원문에 인증키가 있는 URL이 포함되지 않게 예외 종류만 기록합니다.
        result['download_error'] = type(exc).__name__

    download_results.append(result)
    print(
        f"{index:02}/{len(filings)} {receipt_no} | HTTP {result['http_status']} "
        f"| {'ZIP 저장' if result['zip_file'] else result['download_error']}"
    )
    time.sleep(0.2)


01/20 20261012000421 | HTTP 200 | ZIP 저장\n02/20 20261012000021 | HTTP 200 | ZIP 저장\n03/20 20261008000706 | HTTP 200 | ZIP 저장\n04/20 20261008800028 | HTTP 200 | ZIP 저장\n05/20 20261008800004 | HTTP 200 | ZIP 저장\n06/20 20261002800981 | HTTP 200 | ZIP 저장\n07/20 20261001000251 | HTTP 200 | ZIP 저장\n08/20 20261001000203 | HTTP 200 | ZIP 저장\n09/20 20260922000524 | HTTP 200 | ZIP 저장\n10/20 20260918800754 | HTTP 200 | ZIP 저장\n11/20 20260918000212 | HTTP 200 | ZIP 저장\n12/20 20260918000191 | HTTP 200 | ZIP 저장\n13/20 20260917000097 | HTTP 200 | ZIP 저장\n14/20 20260914000248 | HTTP 200 | ZIP 저장\n15/20 20260909000516 | HTTP 200 | ZIP 저장\n16/20 20260909000113 | HTTP 200 | ZIP 저장\n17/20 20260908800624 | HTTP 200 | ZIP 저장\n18/20 20260831000250 | HTTP 200 | ZIP 저장\n19/20 20260831000066 | HTTP 200 | ZIP 저장\n20/20 20260828001916 | HTTP 200 | ZIP 저장\n

In [ ]:
# 모든 본문 내용을 original_content 하나에 규칙별 블록으로 정리합니다.
results = []
content_fields_to_remove = {
    'first_text_block', 'first_table_preview', 'tag_rule_counts',
    'cover', 'cover_locations', 'paragraph_locations', 'body_location',
    'table_locations', 'page_break_locations', 'page_marker_count',
    'page_count_estimate', 'paragraph_count', 'table_count', 'text_source',
    'original_content',
}

for downloaded in download_results:
    result = {key: value for key, value in downloaded.items() if key not in content_fields_to_remove}
    result.update({
        'xml_file': None,
        'xml_bytes': 0,
        'original_content': [],
        'parser_recovery_errors': None,
        'result': 'failed',
        'extraction_error': downloaded.get('download_error'),
    })

    if not downloaded.get('zip_file'):
        results.append(result)
        continue

    try:
        zip_path = ROOT / downloaded['zip_file']
        with zipfile.ZipFile(zip_path) as archive:
            xml_names = [name for name in archive.namelist() if name.lower().endswith('.xml')]
            if not xml_names:
                raise RuntimeError('ZIP 안에 XML 파일이 없습니다.')
            xml_bytes = archive.read(xml_names[0])

        xml_path = OUTPUT_DIR / f"D03_document_{downloaded['rcept_no']}.xml"
        xml_path.write_bytes(xml_bytes)
        parser = etree.XMLParser(recover=True, huge_tree=True)
        document = etree.fromstring(xml_bytes, parser)
        tree = document.getroottree()
        body_nodes = document.xpath(tag_xpath('body'))
        content_root = body_nodes[0] if body_nodes else document

        # PGBRK 전후로 논리 페이지를 배정합니다. 마커가 없으면 page_no는 None입니다.
        page_by_node = {}
        current_page = 1
        page_markers_exist = False
        for node in content_root.iter():
            if not isinstance(node.tag, str):
                continue
            page_by_node[node] = current_page
            if node.tag.split('}')[-1].lower() == 'pgbrk':
                page_markers_exist = True
                current_page += 1

        def page_of(node):
            return page_by_node.get(node) if page_markers_exist else None

        blocks_by_node = {}
        table_nodes = content_root.xpath(tag_xpath('table'))

        # 규칙 1: 표를 행·셀 단위로 묶고 표지 안의 표도 같은 구조로 포함합니다.
        for table_no, table in enumerate(table_nodes, start=1):
            table_xpath = tree.getpath(table)
            table_rows = []
            page_values = [page_by_node.get(table, 1)]
            for row in table.xpath(tag_xpath('tr')):
                nearest_table = row.xpath(
                    'ancestor::*[translate(local-name(), "ABCDEFGHIJKLMNOPQRSTUVWXYZ", "abcdefghijklmnopqrstuvwxyz")="table"][1]'
                )
                if not nearest_table or nearest_table[0] != table:
                    continue
                cells = []
                for cell in row:
                    if not isinstance(cell.tag, str):
                        continue
                    cell_tag = cell.tag.split('}')[-1].lower()
                    if cell_tag not in ('td', 'th', 'tu'):
                        continue
                    page_values.append(page_by_node.get(cell, page_by_node.get(row, 1)))
                    cells.append({
                        'column_no': len(cells) + 1,
                        'tag': cell_tag.upper(),
                        'page_no': page_of(cell),
                        'xpath': tree.getpath(cell),
                        'text': clean_text(cell),
                    })
                if cells:
                    page_values.append(page_by_node.get(row, 1))
                    table_rows.append({
                        'tag': 'TR',
                        'row_no': len(table_rows) + 1,
                        'page_no': page_of(row),
                        'xpath': tree.getpath(row),
                        'cells': cells,
                    })
            if table_rows:
                start_page, end_page = min(page_values), max(page_values)
                blocks_by_node[table] = {
                    'type': 'table',
                    'tag': 'TABLE',
                    'page_start': start_page if page_markers_exist else None,
                    'page_end': end_page if page_markers_exist else None,
                    'xpath': table_xpath,
                    'rows': table_rows,
                }

        # 규칙 2: 표 안의 P는 셀 텍스트에 포함하므로 중복하지 않고, 표 밖 P를 문단으로 저장합니다.
        paragraph_nodes = content_root.xpath(tag_xpath('p'))
        for paragraph in paragraph_nodes:
            if not clean_text(paragraph):
                continue
            inside_table = paragraph.xpath(
                'ancestor::*[translate(local-name(), "ABCDEFGHIJKLMNOPQRSTUVWXYZ", "abcdefghijklmnopqrstuvwxyz")="table"][1]'
            )
            if inside_table:
                continue
            blocks_by_node[paragraph] = {
                'type': 'paragraph',
                'tag': 'P',
                'page_no': page_of(paragraph),
                'xpath': tree.getpath(paragraph),
                'text': clean_text(paragraph),
            }

        # 규칙 3: 표나 P에 포함되지 않은 제목·라벨 텍스트도 원문 순서대로 포함합니다.
        for node in content_root.iter():
            if not isinstance(node.tag, str):
                continue
            tag = node.tag.split('}')[-1].lower()
            if tag in ('body', 'p', 'table', 'tr', 'td', 'th', 'tu', 'pgbrk'):
                continue
            if node.xpath(
                'ancestor::*[translate(local-name(), "ABCDEFGHIJKLMNOPQRSTUVWXYZ", "abcdefghijklmnopqrstuvwxyz")="table"][1]'
            ):
                continue
            if node.xpath(
                'ancestor::*[translate(local-name(), "ABCDEFGHIJKLMNOPQRSTUVWXYZ", "abcdefghijklmnopqrstuvwxyz")="p"][1]'
            ):
                continue
            own_parts = [node.text or '']
            own_parts.extend(child.tail or '' for child in node)
            own_text = ' '.join(' '.join(own_parts).split())
            if own_text:
                blocks_by_node[node] = {
                    'type': 'text',
                    'tag': node.tag.split('}')[-1],
                    'page_no': page_of(node),
                    'xpath': tree.getpath(node),
                    'text': own_text,
                }

        # 하나의 original_content 배열 안에 XML 순서대로 모든 콘텐츠 블록을 둡니다.
        original_content = [blocks_by_node[node] for node in content_root.iter() if node in blocks_by_node]
        if not original_content:
            original_content = [{
                'type': 'text',
                'tag': content_root.tag.split('}')[-1],
                'page_no': page_of(content_root),
                'xpath': tree.getpath(content_root),
                'text': clean_text(content_root),
            }]

        has_content = any(
            block.get('text') or any(cell.get('text') for row in block.get('rows', []) for cell in row.get('cells', []))
            for block in original_content
        )
        result.update({
            'xml_file': str(xml_path.relative_to(ROOT)),
            'xml_bytes': len(xml_bytes),
            'original_content': original_content,
            'parser_recovery_errors': len(parser.error_log),
            'result': 'success' if has_content else 'partial',
            'extraction_error': None if has_content else '원문에서 텍스트를 찾지 못했습니다.',
        })
    except Exception as exc:
        result['extraction_error'] = type(exc).__name__

    results.append(result)
    print(
        f"{len(results):02}/{len(download_results)} {result['rcept_no']} | "
        f"콘텐츠 블록 {len(result['original_content'])} | {result['result']}"
    )

summary = {
    'source_file': str(D02_FILE.relative_to(ROOT)),
    'requested_count': len(filings),
    'success_count': sum(row['result'] == 'success' for row in results),
    'partial_count': sum(row['result'] == 'partial' for row in results),
    'failed_count': sum(row['result'] == 'failed' for row in results),
    'content_field': 'original_content',
    'content_format': '원문 순서의 문단·표·기타 텍스트 블록을 original_content 하나에 배열로 저장합니다. 각 블록에 태그와 위치를 둡니다.',
    'results': results,
}
summary_path = OUTPUT_DIR / 'D03_batch_extraction_20.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('배치 결과 저장:', summary_path.relative_to(ROOT))
print('완전 성공:', summary['success_count'], '| 부분 추출:', summary['partial_count'], '| 실패:', summary['failed_count'])


# 단일 키 원문 정리 방식

- 모든 내용은 `original_content` 하나에 원문 순서의 블록 배열로 저장합니다.
- `type=paragraph`는 문단, `type=table`은 행·셀 구조가 있는 표, `type=text`는 그 밖의 텍스트입니다.
- 각 블록과 표의 행·셀에는 태그·XPath·페이지 위치를 기록합니다. 별도 표지·문단·표 키는 만들지 않습니다.
- ZIP/XML은 접수번호별로 저장하고 전체 결과는 `dart_results/D03_batch_extraction_20.json`에 기록합니다.
